In [13]:
import pandas as pd

df = pd.read_csv("data/submission_template.csv")
print(df.head(5))

    Latitude  Longitude Sample Date  Total Alkalinity  Electrical Conductance  \
0 -32.043333  27.822778  01-09-2014               NaN                     NaN   
1 -33.329167  26.077500  16-09-2015               NaN                     NaN   
2 -32.991639  27.640028  07-05-2015               NaN                     NaN   
3 -34.096389  24.439167  07-02-2012               NaN                     NaN   
4 -32.000556  28.581667  01-10-2014               NaN                     NaN   

   Dissolved Reactive Phosphorus  
0                            NaN  
1                            NaN  
2                            NaN  
3                            NaN  
4                            NaN  


In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
target = ['Total Alkalinity','Electrical Conductance','Dissolved Reactive Phosphorus']

quantitative = [f for f in df.columns if df.dtypes[f] != 'object']
for i in target:
    quantitative.remove((i))

# Get categorical features
categorical = [f for f in df.columns if df.dtypes[f] == 'object']

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for i in target:
    plt.figure(figsize=(10,6))
    sns.distplot(df[i])
    plt.title('Histogram of %s' % i)
    plt.show()

## Joining three files together

In [ ]:
import pandas as pd

def join_and_identify_mismatches(wq_path, terra_path, landsat_path):
    # 1. Load the datasets
    wq_df = pd.read_csv(wq_path)
    terra_df = pd.read_csv(terra_path)
    landsat_df = pd.read_csv(landsat_path)

    # Helper function to standardize keys
    def clean_and_format(df):
        df.columns = df.columns.str.strip()
        # FIX: Handle day-first date format
        df['Sample Date'] = pd.to_datetime(df['Sample Date'], dayfirst=True)
        # Round coordinates to ensure they match accurately
        df['Latitude'] = df['Latitude'].round(5)
        df['Longitude'] = df['Longitude'].round(5)
        return df

    wq_df = clean_and_format(wq_df)
    terra_df = clean_and_format(terra_df)
    landsat_df = clean_and_format(landsat_df)

    # 2. Join Water Quality with TerraClimate
    # Using 'outer' to keep all records for debugging/reporting
    merged = pd.merge(
        wq_df, terra_df,
        on=['Latitude', 'Longitude', 'Sample Date'],
        how='inner',
        indicator='_merge_terra'
    )

    # 3. Join with Landsat
    final_merged = pd.merge(
        merged, landsat_df,
        on=['Latitude', 'Longitude', 'Sample Date'],
        how='inner',
        indicator='_merge_landsat'
    )

    # 4. Count Unmerged Records from the perspective of Water Quality
    # Only records that exist in Water Quality but are missing from features
    wq_only = final_merged[~final_merged['Total Alkalinity'].isna()]

    # Missing from TerraClimate
    missing_terra = wq_only[wq_only['_merge_terra'] == 'left_only']

    # Missing from Landsat
    missing_landsat = wq_only[wq_only['_merge_landsat'] == 'left_only']

    # Missing from BOTH simultaneously
    missing_both = wq_only[(wq_only['_merge_terra'] == 'left_only') &
                           (wq_only['_merge_landsat'] == 'left_only')]

    # Missing from AT LEAST ONE feature set
    missing_either = wq_only[(wq_only['_merge_terra'] == 'left_only') |
                             (wq_only['_merge_landsat'] == 'left_only')]

    print(f"Total Water Quality records: {len(wq_df)}")
    print(f"1. Missing TerraClimate: {len(missing_terra)}")
    print(f"2. Missing Landsat: {len(missing_landsat)}")
    print(f"3. Missing BOTH at the same time: {len(missing_both)}")
    print(f"4. Total records with at least one missing feature: {len(missing_either)}")

    # Optional: Save the merged data (without dropping NaNs)
    final_merged.to_csv('@merged_data_unfiltered.csv', index=False)

    return final_merged


result = join_and_identify_mismatches('submission_template.csv',
                                       'terraclimate_features_validation.csv',
                                      'landsat_features_validation.csv')

In [ ]:
train = result

In [ ]:
train.info()

In [ ]:
train.head()

Now we found that there are some missing data in Landsat Data. Let's try to fetch from api and impute first. If we can't may be we need to use KNN

## Landsat API fetching

In [ ]:
!pip install pystac_client planetary_computer rasterio pyproj

In [ ]:
# rioxarray no longer needed — using rasterio directly

In [ ]:
import pystac_client
import planetary_computer
import rasterio
from pyproj import Transformer
import numpy as np
from datetime import timedelta

# Connect to the catalog
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

def read_pixel(href, lon, lat):
    """
    Read a single pixel from a Cloud-Optimized GeoTIFF using proper CRS transform.

    The GeoTIFF is in UTM (meters), but our coordinates are in lon/lat (degrees).
    We use pyproj to convert before reading.
    """
    with rasterio.open(href) as src:
        transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
        x_proj, y_proj = transformer.transform(lon, lat)
        row, col = src.index(x_proj, y_proj)

        if not (0 <= row < src.height and 0 <= col < src.width):
            return None

        window = rasterio.windows.Window(col, row, 1, 1)
        value = src.read(1, window=window)[0, 0]
        return int(value)

def is_pixel_clear(qa_value):
    """
    Decodes the Landsat QA Pixel.
    Bit 3 = Cloud, Bit 4 = Cloud Shadow.
    Returns True if both are 0 (clear).
    """
    is_cloud = (qa_value >> 3) & 1
    is_shadow = (qa_value >> 4) & 1
    return (is_cloud == 0) and (is_shadow == 0)

def get_landsat_pixel_from_api(lat, lon, date_str):
    """
    Fetches specific pixel values for a coordinate from the Planetary Computer.

    Handles:
    - CRS transformation (lon/lat -> UTM)
    - Cloud masking via QA band
    - Landsat 7 SLC-off stripe skipping (value == 0)
    - Tiered search: narrow time window first, then widen
    """
    target_date = pd.to_datetime(date_str)

    # Tiers: (Days window, Scene Cloud Max %)
    tiers = [(8, 20), (16, 40), (32, 60)]

    point = {"type": "Point", "coordinates": [lon, lat]}

    for days, scene_cloud in tiers:
        start = (target_date - timedelta(days=days)).strftime('%Y-%m-%d')
        end = (target_date + timedelta(days=days)).strftime('%Y-%m-%d')

        search = catalog.search(
            collections=["landsat-c2-l2"],
            intersects=point,
            datetime=f"{start}/{end}",
            query={"eo:cloud_cover": {"lt": scene_cloud}}
        )

        items = list(search.items())
        # Sort by closeness to target date
        items.sort(key=lambda x: abs((x.datetime.replace(tzinfo=None) - target_date).days))

        for item in items:
            try:
                # --- STEP 1: Cloud Check ---
                qa_href = planetary_computer.sign(item.assets["qa_pixel"].href)
                qa_val = read_pixel(qa_href, lon, lat)

                if qa_val is None or not is_pixel_clear(qa_val):
                    continue

                # --- STEP 2: Check NIR for SLC stripe / fill (value == 0) ---
                nir_href = planetary_computer.sign(item.assets["nir08"].href)
                nir_val = read_pixel(nir_href, lon, lat)

                if nir_val is None or nir_val == 0:
                    continue  # SLC stripe or fill — try next scene

                # --- STEP 3: Extract all bands ---
                band_map = {
                    'nir': 'nir08',
                    'green': 'green',
                    'swir16': 'swir16',
                    'swir22': 'swir22'
                }

                data = {}
                all_good = True
                for my_name, asset_key in band_map.items():
                    href = planetary_computer.sign(item.assets[asset_key].href)
                    val = read_pixel(href, lon, lat)
                    if val is None or val == 0:
                        all_good = False
                        break
                    data[my_name] = val

                if all_good:
                    return data, f"API_{days}d"

            except Exception as e:
                continue

    return None, "KNN"


In [ ]:
from sklearn.impute import KNNImputer

class KNNFallbackEngine:
    def __init__(self, n_neighbors=5):
        self.imputer = KNNImputer(n_neighbors=n_neighbors)
        self.features = ['Latitude', 'Longitude', 'pet']
        self.targets = ['nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI']

    def fit(self, df):
        """Train the imputer on rows that HAVE data."""
        clean_data = df.dropna(subset=self.targets)
        self.imputer.fit(clean_data[self.features + self.targets])
        print(f"KNN Engine fitted on {len(clean_data)} rows.")

    def fill(self, df_subset):
        """Predict values for rows that are STILL missing data."""
        # Note: KNN needs all columns present even if they are NaN
        imputed_values = self.imputer.transform(df_subset[self.features + self.targets])
        return pd.DataFrame(imputed_values, columns=self.features + self.targets, index=df_subset.index)

In [ ]:
def run_imputation_pipeline(df):
    # 1. Prepare KNN Engine
    knn_engine = KNNFallbackEngine()
    knn_engine.fit(df)

    # 2. API IMPUTATION
    missing_indices = df[df['nir'].isna()].index
    print(f"Starting API fetch for {len(missing_indices)} missing rows...")

    for idx in missing_indices:
        row = df.loc[idx]
        # UNPACK BOTH VALUES: the dictionary and the method string
        api_data, method = get_landsat_pixel_from_api(row['Latitude'], row['Longitude'], row['Sample Date'])

        if api_data:
            # FIXED: Use 'nir', not 'nir08' to match what the function returns
            df.at[idx, 'nir'] = api_data['nir']
            df.at[idx, 'green'] = api_data['green']
            df.at[idx, 'swir16'] = api_data['swir16']
            df.at[idx, 'swir22'] = api_data['swir22']

            # CALCULATE INDICES: Since we have the raw numbers,
            # we should update the NDMI/MNDWI columns now too.
            n, g, s = api_data['nir'], api_data['green'], api_data['swir16']
            df.at[idx, 'NDMI'] = (n - s) / (n + s) if (n + s) != 0 else 0
            df.at[idx, 'MNDWI'] = (g - s) / (g + s) if (g + s) != 0 else 0

            df.at[idx, 'Impute_Method'] = method

    # 3. KNN FALLBACK
    still_missing = df[df['nir'].isna()]
    if not still_missing.empty:
        print(f"API failed for {len(still_missing)} rows. Switching to KNN...")
        imputed_df = knn_engine.fill(still_missing)

        for col in knn_engine.targets:
            df.loc[still_missing.index, col] = imputed_df[col]
        df.loc[still_missing.index, 'Impute_Method'] = 'KNN'

    return df

In [ ]:
# 1. Create your test subset (First 50 rows)
train_subset = train[train['NDMI'].isna()].head(50).copy()
train_subset['Impute_Method'] = 'Original'

# 2. Initialize and Fit KNN on FULL data
knn_engine = KNNFallbackEngine()
knn_engine.fit(train)

# --- Step A: API Search ---
missing_indices = train_subset[train_subset['nir'].isna()].index
print(f"Checking {len(missing_indices)} missing rows in the top 50...")

for idx in missing_indices:
    row = train_subset.loc[idx]

    # Use the function name exactly as defined in the corrected version
    api_data, method = get_landsat_pixel_from_api(row['Latitude'], row['Longitude'], row['Sample Date'])

    if api_data:
        print(f"  -> Found data for Row {idx}")
        # FIXED: Use the dictionary keys returned by the function ('nir', not 'nir08')
        train_subset.at[idx, 'nir']   = api_data['nir']
        train_subset.at[idx, 'green'] = api_data['green']
        train_subset.at[idx, 'swir16'] = api_data['swir16']
        train_subset.at[idx, 'swir22'] = api_data['swir22']

        # CALCULATE INDICES: Since we have real numbers now, let's build the indices
        # This prevents the model from seeing NaN in the index columns
        n = api_data['nir']
        s = api_data['swir16']
        g = api_data['green']

        train_subset.at[idx, 'NDMI'] = (n - s) / (n + s) if (n + s) != 0 else 0
        train_subset.at[idx, 'MNDWI'] = (g - s) / (g + s) if (g + s) != 0 else 0

        train_subset.at[idx, 'Impute_Method'] = method
    else:
        print(f"  -> No data for Row {idx}. Marking for KNN.")

# --- Step B: KNN Fill ---
still_missing = train_subset[train_subset['nir'].isna()]

if not still_missing.empty:
    print(f"Refining {len(still_missing)} rows with KNN...")
    imputed_data = knn_engine.fill(still_missing)

    for col in knn_engine.targets:
        train_subset.loc[still_missing.index, col] = imputed_data[col]
    train_subset.loc[still_missing.index, 'Impute_Method'] = 'KNN'

# 4. Save
train_subset.to_csv("train_test_50_pipeline.csv", index=False)

In [ ]:
train_subset

In [ ]:
# Assuming 'train' is your merged dataframe
train['Impute_Method'] = 'Original' # Track where data comes from
train_final = run_imputation_pipeline(train)

# Save your work
train_final.to_csv("data/train_complete_pipeline.csv", index=False)

In [ ]:
train_final.head(20)